In [1]:
# ── Cell 0: Fix PyTorch for P100 (compute 6.0) ────────────────────────────
!pip install torch==2.4.0 torchvision==0.19.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers pyarrow tqdm -q
print('Done!')

import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Compute:', torch.cuda.get_device_capability(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 110.7 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 75.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 48.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 103.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 1.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 6.7 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 14.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 9.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━

In [2]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(
        torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('WARNING: No GPU!')

CUDA available: True
GPU: Tesla P100-PCIE-16GB
VRAM: 17.1 GB


In [3]:
# ── Cell 2: Verify files ───────────────────────────────────────────────────
import os

MODEL_PATH   = '/kaggle/input/datasets/yogendra2005/blair-custom-model/blair-videogames-multiaspect'
PARQUET_PATH = '/kaggle/input/datasets/yogendra2005/blair-products-rich/products_rich.parquet'

print('Model files:')
for f in os.listdir(MODEL_PATH):
    size = os.path.getsize(f'{MODEL_PATH}/{f}') / 1024 / 1024
    print(f'  {f}: {size:.1f} MB')

print(f'\nParquet exists: {os.path.exists(PARQUET_PATH)}')
print('\nAll files ready!')

Model files:
  config.json: 0.0 MB
  tokenizer.json: 3.4 MB
  tokenizer_config.json: 0.0 MB
  model.safetensors: 1355.6 MB

Parquet exists: True

All files ready!


In [4]:
# ── Cell 3: Generate item embeddings ───────────────────────────────────────
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

# Config
BATCH_SIZE       = 64
MAX_LENGTH       = 512
EMB_DIM          = 1024
CHECKPOINT_EVERY = 5000
OUT_EMB          = '/kaggle/working/item_embeddings.npy'
OUT_IDS          = '/kaggle/working/item_ids.npy'
PARTIAL_EMB      = '/kaggle/working/item_embeddings_partial.npy'
LAST_IDX_FILE    = '/kaggle/working/last_index.txt'

# Load data
df      = pd.read_parquet(PARQUET_PATH)
texts   = df['rich_text'].fillna('').tolist()
asins   = df['parent_asin'].tolist()
n_items = len(texts)
print(f'Loaded {n_items:,} products')

# Device — use cuda (PyTorch 2.4 supports P100)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load model
print('Loading model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModel.from_pretrained(MODEL_PATH).to(device).eval()
print('Model ready!')

# Test GPU works before starting loop
print('Testing GPU...')
test_enc = tokenizer(['test'], return_tensors='pt',
                     truncation=True, max_length=16)
test_enc = {k: v.to(device) for k, v in test_enc.items()}
with torch.no_grad():
    test_out = model(**test_enc)
print('GPU test passed!')

# Resume checkpoint if exists
start_idx  = 0
embeddings = np.zeros((n_items, EMB_DIM), dtype=np.float32)
if Path(PARTIAL_EMB).exists() and Path(LAST_IDX_FILE).exists():
    start_idx = int(Path(LAST_IDX_FILE).read_text().strip())
    partial   = np.load(PARTIAL_EMB)
    if partial.shape == (start_idx, EMB_DIM):
        embeddings[:start_idx] = partial
        print(f'Resumed from checkpoint: {start_idx}/{n_items}')
    else:
        start_idx = 0
        print('Checkpoint mismatch — starting fresh')

# Embedding loop
t0 = time.time()
for batch_start in tqdm(
        range(start_idx, n_items, BATCH_SIZE), desc='Embedding'):

    batch_end   = min(batch_start + BATCH_SIZE, n_items)
    batch_texts = texts[batch_start:batch_end]

    enc = tokenizer(
        batch_texts,
        max_length=MAX_LENGTH,
        padding=True,
        truncation=True,
        return_tensors='pt'
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)

    cls = out.last_hidden_state[:, 0, :]
    cls = F.normalize(cls, dim=-1)
    embeddings[batch_start:batch_end] = cls.cpu().float().numpy()

    # Checkpoint every 5000 items
    if batch_end % CHECKPOINT_EVERY < BATCH_SIZE:
        np.save(PARTIAL_EMB, embeddings[:batch_end])
        Path(LAST_IDX_FILE).write_text(str(batch_end))
        elapsed = time.time() - t0
        rate    = batch_end / elapsed
        eta     = (n_items - batch_end) / rate / 60
        print(f'Checkpoint: {batch_end}/{n_items} | '
              f'{rate:.0f} items/s | ETA {eta:.1f} min')

elapsed = time.time() - t0
print(f'\nDone: {n_items:,} items in {elapsed/60:.1f} min')

# Save
np.save(OUT_EMB, embeddings)
np.save(OUT_IDS, np.array(asins, dtype=object))
print(f'Saved item_embeddings.npy shape={embeddings.shape}')
print(f'Saved item_ids.npy count={len(asins)}')

# Cleanup
for f in [PARTIAL_EMB, LAST_IDX_FILE]:
    if Path(f).exists(): Path(f).unlink()

Loaded 137,269 products
Device: cuda
Loading model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model ready!
Testing GPU...
GPU test passed!


Embedding:   4%|▎         | 79/2145 [04:27<1:56:55,  3.40s/it]

Checkpoint: 5056/137269 | 19 items/s | ETA 116.7 min


...

Checkpoint: 130048/137269 | 19 items/s | ETA 6.4 min


Embedding:  98%|█████████▊| 2110/2145 [1:59:16<02:02,  3.51s/it]

Checkpoint: 135040/137269 | 19 items/s | ETA 2.0 min


Embedding: 100%|██████████| 2145/2145 [2:01:14<00:00,  3.39s/it]



Done: 137,269 items in 121.2 min
Saved item_embeddings.npy shape=(137269, 1024)
Saved item_ids.npy count=137269


In [5]:
# ── Cell 4: Verify ─────────────────────────────────────────────────────────
import numpy as np

emb = np.load('/kaggle/working/item_embeddings.npy')
ids = np.load('/kaggle/working/item_ids.npy', allow_pickle=True)

print('Shape:', emb.shape)
print('IDs:', len(ids))
norms = np.linalg.norm(emb, axis=1)
print(f'Norm mean: {norms.mean():.6f}')
print(f'Norm min:  {norms.min():.6f}')

assert emb.shape == (137269, 1024), f'Wrong shape: {emb.shape}'
assert len(ids) == 137269, f'Wrong count: {len(ids)}'
assert np.allclose(norms, 1.0, atol=1e-4), 'Not normalized!'
print('\nALL CHECKS PASSED! ✅')
print('Download from Output tab on right →')

Shape: (137269, 1024)
IDs: 137269
Norm mean: 1.000000
Norm min:  1.000000

ALL CHECKS PASSED! ✅
Download from Output tab on right →


In [6]:
# ── Cell 5: Download ───────────────────────────────────────────────────────
# Files are in /kaggle/working/
# Download from Output tab on right panel:
#   item_embeddings.npy (~536MB)
#   item_ids.npy (~1.7MB)

print('Files ready to download:')
import os
for f in ['item_embeddings.npy', 'item_ids.npy']:
    path = f'/kaggle/working/{f}'
    if os.path.exists(path):
        size = os.path.getsize(path)/1024/1024
        print(f'  {f}: {size:.1f} MB')

Files ready to download:
  item_embeddings.npy: 536.2 MB
  item_ids.npy: 1.7 MB
